#**[Wildfire Prediction using Time Series Analysis](https://www.sciencebuddies.org/)**

This notebook was developed by Science Buddies [www.sciencebuddies.org](https://www.sciencebuddies.org/) as part of a science project to allow students to explore and learn about artificial intelligence. For personal use, this notebook can be downloaded and modified with attribution. For all other uses, please see our [Terms and Conditions of Fair Use](https://www.sciencebuddies.org/about/terms-and-conditions-of-fair-use).  

**Troubleshooting tips**
*   Read the written instructions at Science Buddies and the text and comments on this page carefully.
*   If you make changes that break the code, you can download a fresh copy of this notebook and start over.

*   If you are using this notebook for a science project and need help, visit our [Ask an Expert](https://www.sciencebuddies.org/science-fair-projects/ask-an-expert-intro) forum for assistance.

## **How To Use This Notebook**

This notebook contains text fields, like this one, that give you information about the project and instructions.

In [ ]:
# There are also code blocks, like this one.

# The green text in a code block are comments. Comments are descriptions of what the code does.

# The non-green text in a code block is the Python code. Click on the triangle in the top left corner to run this code block.

print("Congratulations, you ran a code block! Try changing the text in the code and running it again.")

# Importing Libraries

In [ ]:
# Install Prophet for forecasting and the Holidays package (v0.28) for holiday features
!pip install prophet holidays==0.28

In [ ]:
# --- Core libraries ---
import warnings                       # To handle warning messages
import glob                           # For working with file paths/patterns

# --- Data manipulation ---
import pandas as pd                   # Data analysis and manipulation
import numpy as np                    # Numerical computations

# --- Visualization ---
import matplotlib.pyplot as plt       # Plotting
import matplotlib.colors as mcolors   # Advanced color handling for matplotlib
import seaborn as sns                 # Statistical plotting (built on matplotlib)
import folium                         # Interactive maps
import branca.colormap as cm          # Colormaps for Folium visualizations

# --- Machine learning / forecasting ---
from prophet import Prophet           # Time series forecasting
from sklearn.metrics import mean_absolute_error  # Model evaluation metric

# --- Pandas extensions ---
from pandas.api.types import CategoricalDtype    # Handling categorical dtypes

# --- Global settings ---
warnings.filterwarnings("ignore")     # Suppress warnings
plt.style.use('ggplot')               # Set plot style
plt.style.use('fivethirtyeight')      # Override with another style

# --- Custom utility functions ---
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates Mean Absolute Percentage Error (MAPE) given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# Import Google Colab's drive module to access Google Drive files
from google.colab import drive

# Mount Google Drive to the Colab environment at the given path (/content/drive)
# After running this, you'll be prompted to authorize access to your Google Drive
drive.mount('/content/drive')

# Loading the Data into a Pandas DataFrame

Code Block 1A

In [ ]:
# Path to the folder containing wildfire CSV files on Google Drive
csv_path = "/content/drive/MyDrive/wildfire_prediction/"

# Find all CSV files in the specified folder and store their file paths in a list
csv_files = glob.glob(csv_path + "*.csv")

# Read each CSV file into a Pandas DataFrame and store all DataFrames in a list
dfs = [pd.read_csv(f) for f in csv_files]

# Concatenate all DataFrames into a single DataFrame, resetting the row index
df = pd.concat(dfs, ignore_index=True)

# Convert the 'acq_date' column to datetime format for easier date operations
df['acq_date'] = pd.to_datetime(df['acq_date'])

# Set 'acq_date' as the index of the DataFrame for time series handling
df.set_index('acq_date', inplace=True)

# Sort the DataFrame by date (index) in ascending order
df.sort_index(inplace=True)

# Save the combined, sorted DataFrame to a new CSV file in the same folder
df.to_csv(csv_path + "df_2019_2024.csv")

Code Block 1B

In [ ]:
# Define the path to the project folder stored in Google Drive
csv_path = "/content/drive/MyDrive/wildfire_prediction/"

# Load the dataset (CSV file) into a pandas DataFrame
# In this case, we are reading "df_2019_2024.csv" from the specified folder
df = pd.read_csv(csv_path + "df_2019_2024.csv")

# Convert the 'acq_date' column to datetime objects and set it as the index
df['acq_date'] = pd.to_datetime(df['acq_date'])
df.set_index('acq_date', inplace=True)

df.head()

# Visualizing the Data

Code Block 2A

In [ ]:
def plot_frp_by_timeframe(df, year, month=None):
    """
    Generates a line plot of Fire Radiative Power (FRP) for a specific year and optional month,
    or a bar graph of monthly median FRP for a specific year.

    Args:
        df (pd.DataFrame): The input DataFrame with a datetime index.
        year (int): The year to filter by.
        month (int, optional): The month to filter by (1-12). If None, a bar graph
                               of monthly median FRP for the year is generated.
    """
    if month is not None:
        df_filtered = df[(df.index.year == year) & (df.index.month == month)].copy()
        timeframe_str = f"Fire Radiative Power (FRP) for {month}/{year}"
        plot_type = 'line'
    else:
        df_filtered = df[df.index.year == year].copy()
        timeframe_str = f"Monthly Median Fire Radiative Power (FRP) for {year}"
        plot_type = 'bar'

    if df_filtered.empty:
        print(f"No data found for {timeframe_str.replace('Monthly Median ', '')}")
        return

    plt.figure(figsize=(12, 6))

    if plot_type == 'line':
        plt.plot(df_filtered.index, df_filtered['frp'])
        plt.xlabel('Date')
        plt.ylabel('FRP')
    elif plot_type == 'bar':
        # Group by month and calculate the mean FRP
        monthly_frp = df_filtered.resample('M')['frp'].mean()
        monthly_frp.index = monthly_frp.index.strftime('%Y-%m') # Format for x-axis
        monthly_frp.plot(kind='bar')
        plt.xlabel('Month')
        plt.ylabel('Median FRP')
        plt.xticks(rotation=45, ha='right') # Rotate labels for better readability

    plt.title(timeframe_str)
    plt.grid(True)
    plt.tight_layout() # Adjust layout to prevent labels overlapping
    plt.show()

Code Block 2B

In [ ]:
# TODO: Change year
year = 2023

 # Bar graph for the year
plot_frp_by_timeframe(df, year)

Code Block 2C

In [ ]:
def plot_intense_fires_folium(df, year, month=None):
    """
    Generates an interactive map showing the locations of intense fires (FRP >= 100)
    for a specific year and optional month using Folium, with color scaling by intensity
    and a legend.

    Args:
        df (pd.DataFrame): The input DataFrame with a datetime index and
                           'latitude', 'longitude', and 'frp' columns.
        year (int): The year to filter by.
        month (int, optional): The month to filter by (1-12). Defaults to None.
    """
    if month is not None:
        df_filtered = df[(df.index.year == year) & (df.index.month == month)].copy()
        timeframe_str = f"Intense Fires for {month}/{year}"
    else:
        df_filtered = df[df.index.year == year].copy()
        timeframe_str = f"Intense Fires for {year}"

    # Filter for intense fires (FRP >= 100)
    intense_fires_df = df_filtered[df_filtered['frp'] >= 100].copy()

    if intense_fires_df.empty:
        print(f"No intense fires found for {timeframe_str}")
        return

    # Create a base map of the US
    m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)

    # Add markers for intense fire locations
    max_frp = intense_fires_df['frp'].max()
    min_frp = intense_fires_df['frp'].min()

    # Create a colormap for FRP values
    cmap = plt.cm.get_cmap('YlOrRd') # Yellow to Red colormap
    norm = mcolors.Normalize(vmin=min_frp, vmax=max_frp)

    for index, row in intense_fires_df.iterrows():
        # Scale FRP to a reasonable range for radius (e.g., 2 to 15)
        radius = 2 + (row['frp'] - min_frp) / (max_frp - min_frp) * 13 if max_frp > min_frp else 5
        # Scale FRP to a reasonable range for opacity (e.g., 0.4 to 0.9)
        opacity = 0.4 + (row['frp'] - min_frp) / (max_frp - min_frp) * 0.5 if max_frp > min_frp else 0.7

        # Get color from colormap based on FRP value
        color = mcolors.to_hex(cmap(norm(row['frp'])))

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=opacity,
            tooltip=f"FRP: {row['frp']:.2f}"
        ).add_to(m)

    # Add a legend
    colormap = cm.LinearColormap(
        colors=['yellow', 'orange', 'red'],
        index=[0, 250, 500],   # evenly spaced points in the range
        vmin=0,
        vmax=500
    )

    # Add the colormap as a legend to the map
    colormap.caption = 'Fire Radiative Power (FRP)'
    colormap.add_to(m)

    # Display the map
    display(m)

In [ ]:
# TODO: Change year and month
year = 2024
month = 7

plot_intense_fires_folium(df, year, month)

# Preprocessing

Code Block 3A

In [ ]:
# List of column names that we don't need for our analysis
columns_to_drop = ['acq_time', 'satellite', 'instrument', 'version', 'daynight']

# Remove the specified columns from the DataFrame "df"
# inplace=True means the change is applied directly to df without creating a new DataFrame
df.drop(columns=columns_to_drop, inplace=True)

# Display the first 5 rows of the updated DataFrame to confirm the columns were removed
df.head()

Code Block 3B

In [ ]:
# TODO: Adjust latitude and longitude to the region you want
min_latitude, max_latitude = 32, 42
min_longitude, max_longitude = -124, -114

# Filter by latitude and longitude bounds
df_filtered = df[(df['latitude'] >= min_latitude) & (df['latitude'] <= max_latitude) & (df['longitude'] >= min_longitude) & (df['longitude'] <= max_longitude)]

print(f"Number of rows after filtering: {len(df_filtered)}")

Code Block 3C

In [ ]:
# Create a map centered around the area
m = folium.Map(location=[(min_latitude + max_latitude) / 2, (min_longitude + max_longitude) / 2], zoom_start=6)

# Add a rectangle representing the boundaries
folium.Rectangle(
    bounds=[(min_latitude, min_longitude), (max_latitude, max_longitude)],
    color='#ff7800',
    weight=2,
    fill=True,
    fill_color='#ffff00',
    fill_opacity=0.2
).add_to(m)

# Display the map
display(m)

Code Block 3D

In [ ]:
# Aggregate by daily, drop latitude and longitude, and median other features
df = df_filtered.resample('D').median()

# Drop NaN values
df.dropna(inplace=True)

# Display the dataset
df.head()

# Visualizing Region Data

Code Block 4A

In [ ]:
# Get the default seaborn color palette (list of colors to use in plots)
color_pal = sns.color_palette()

# Plot the 'frp' column from the DataFrame
df.plot(
    style='.',                # Plot style: '.' means scatter-style points
    y='frp',                  # Column to plot on the y-axis
    figsize=(10, 5),          # Set the figure size (width=10, height=5)
    ms=1,                     # Marker size (small points since the dataset may be large)
    color=color_pal[0],       # Use the first color from the seaborn palette
    title='Wildfire Data',    # Title for the plot
    alpha=0.5                 # Set transparency (helps reduce overplotting)
)

# Label the x-axis and y-axis
plt.xlabel('Date')            # X-axis label: dates (time series)
plt.ylabel('FRP')             # Y-axis label: Fire Radiative Power

Code Block 4B

In [ ]:
def create_features(df):
    """
    Creates time series features from datetime index and concatenates to original df.
    """
    df = df.copy()
    df['date'] = df.index
    df['month'] = df['date'].dt.month
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['date_offset'] = (df.date.dt.month*100 + df.date.dt.day - 320)%1300

    df['season'] = pd.cut(df['date_offset'], [0, 300, 602, 900, 1300],
                          labels=['Spring', 'Summer', 'Fall', 'Winter']
                   )
    df = df.drop(columns=['date', 'date_offset']) # Drop temporary columns

    return df

df_features = create_features(df)
X = df_features.drop(columns=['frp'])
y = df_features['frp']

Code Block 4C

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10)) # Increased figure size
sns.boxplot(data=df_features.dropna(),
            x='season',
            y='frp',
            hue='season',
            ax=ax,
            linewidth=0.5) # Slightly reduced linewidth
ax.set_title('FRP by Day of Year and Season')
ax.set_xlabel('Season')
ax.set_ylabel('FRP')
ax.legend(bbox_to_anchor=(1, 1))
plt.show()

# Splitting Data into Train and Test

Code Block 5A

In [ ]:
# TODO: Change date
split_date = '1-Jan-2024'
df_train = df_features.loc[df_features.index <= split_date].copy()
df_test = df_features.loc[df_features.index > split_date].copy()

# Plot train and test so you can see where we have split
pd.concat([df_test.groupby(df_test.index)['frp'].mean().rename('TEST SET'),
           df_train.groupby(df_train.index)['frp'].mean().rename('TRAINING SET')],
          axis=1) \
    .plot(figsize=(10, 5), title='Wildfire FRP', style='.', ms=1)
plt.show()

# Training the Model

Code Block 6A

In [ ]:
# Format data for prophet model using ds and y
df_train_prophet = df_train.reset_index() \
    .rename(columns={'acq_date':'ds',
                     'frp':'y'})

Code Block 6B

In [ ]:
%%time
# Initialize and train the Prophet model
# The Prophet model is a time series forecasting model developed by Facebook
model = Prophet()
# Fit the model to the training data
# This step can take some time as the model learns the patterns in the data
model.fit(df_train_prophet)

# Evaluating the Model

Code Block 7A

In [ ]:
# Predict on test set with model
df_test_prophet = df_test.reset_index() \
    .rename(columns={'acq_date':'ds',
                     'frp':'y'})

# Use the trained Prophet model to make predictions on the test data
df_test_fcst = model.predict(df_test_prophet)

Code Block 7B

In [ ]:
# Create a figure and axis for plotting, setting the figure size
fig, ax = plt.subplots(figsize=(10, 5))

# Plot the Prophet forecast results on the specified axis
fig = model.plot(df_test_fcst, ax=ax)

# Plot the actual values from the test set as scatter points
ax.scatter(df_test_prophet['ds'], df_test_prophet['y'], color='black', s=5)

# Set the plot title
ax.set_title('Prophet Forecast vs Actual')
yaxis = ax.get_yaxis()
yaxis.set_label_text('FRP')

# Add a legend to differentiate between forecast and actual values
ax.legend()

# Display the plot
plt.show()

Code Block 7C

In [ ]:
# Make a graph to compare prediction to actual values with rolling median for the test set
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_test_fcst['ds'], df_test_fcst['yhat'], label='Prophet Forecast', alpha=0.7)
ax.plot(df_test_prophet['ds'], df_test_prophet['y'], label='Actual', alpha=0.7)

# TODO: You can adjust the window size
rolling_window = 28

# Calculate and plot rolling median
df_test_prophet['rolling_avg'] = df_test_prophet['y'].rolling(window=rolling_window).median()
ax.plot(df_test_prophet['ds'], df_test_prophet['rolling_avg'], label=f'Rolling Median ({rolling_window} days)', alpha=0.7)

ax.set_title('Prophet Forecast vs Actual with Rolling Median (2024)')
ax.set_xlabel('Date')
ax.set_ylabel('FRP')
ax.legend()
plt.show()

Code Block 7D

In [ ]:
# Calculate MAE
mae = mean_absolute_error(df_test['frp'], df_test_fcst['yhat'])
print('MAE:', mae)

Code Block 7E

In [ ]:
# Calculate MAPE
mean_absolute_percentage_error(df_test['frp'], df_test_fcst['yhat'])

Code Block 7F

In [ ]:
def plot_prophet_predictions_folium(df_fcst, year, month=None):
    """
    Generates an interactive map showing the locations of Prophet model predictions
    for a specific year and optional month using Folium, with color scaling by predicted FRP.

    Args:
        df_fcst (pd.DataFrame): The input DataFrame with 'ds', 'yhat', 'latitude',
                                'longitude' columns (from Prophet forecast and original data).
        year (int): The year to filter by.
        month (int, optional): The month to filter by (1-12). Defaults to None.
    """
    # Ensure 'ds' is datetime and set as index for filtering
    df_fcst['ds'] = pd.to_datetime(df_fcst['ds'])
    df_fcst.set_index('ds', inplace=True)

    if month is not None:
        df_filtered = df_fcst[(df_fcst.index.year == year) & (df_fcst.index.month == month)].copy()
        timeframe_str = f"Prophet Predictions for {month}/{year}"
    else:
        df_filtered = df_fcst[df_fcst.index.year == year].copy()
        timeframe_str = f"Prophet Predictions for {year}"

    # Drop rows with NaN in latitude or longitude
    df_filtered.dropna(subset=['latitude', 'longitude'], inplace=True)

    if df_filtered.empty:
        print(f"No predictions with location data found for {timeframe_str}")
        return

    # Create a base map of the US
    m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)

    # Add markers for predicted fire locations
    # Use 'yhat' for coloring and radius
    max_yhat = df_filtered['yhat'].max()
    min_yhat = df_filtered['yhat'].min()

    # Create a colormap for yhat values
    cmap = plt.cm.get_cmap('YlOrRd') # Yellow to Red colormap
    norm = mcolors.Normalize(vmin=min_yhat, vmax=max_yhat)

    for index, row in df_filtered.iterrows():
        # Scale yhat to a reasonable range for radius (e.g., 2 to 15)
        radius = 2 + (row['yhat'] - min_yhat) / (max_yhat - min_yhat) * 13 if max_yhat > min_yhat else 5
        # Scale yhat to a reasonable range for opacity (e.g., 0.4 to 0.9)
        opacity = 0.4 + (row['yhat'] - min_yhat) / (max_yhat - min_yhat) * 0.5 if max_yhat > min_yhat else 0.7

        # Get color from colormap based on yhat value
        color = mcolors.to_hex(cmap(norm(row['yhat'])))

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=opacity,
            tooltip=f"Predicted FRP: {row['yhat']:.2f}"
        ).add_to(m)


      # Add a legend
    colormap = cm.LinearColormap(
        colors=['yellow', 'orange', 'red'],
        index=[0, 250, 500],   # evenly spaced points in the range
        vmin=0,
        vmax=500
    )

    # Add the colormap as a legend to the map
    colormap.caption = 'Fire Radiative Power (FRP)'
    colormap.add_to(m)

    # Display the map
    display(m)

Code Block 7G

In [ ]:
# TODO: Change year and month
year = 2024
month = 7

# Merge latitude and longitude from df_test_prophet into df_test_fcst
df_test_fcst_merged = df_test_fcst.merge(df_test_prophet[['ds', 'latitude', 'longitude']], on='ds', how='left')

plot_prophet_predictions_folium(df_test_fcst_merged, year, month)

Code Block 7H

In [ ]:
# TODO: Change year and month
year = 2024
month = 7

# Plot actual fires
plot_intense_fires_folium(df, year)